## Importation des bibliothèques nécessaires et initialiser du dataframee

In [1]:
import whisper
import pandas as pd
import os

100%|███████████████████████████████████████| 139M/139M [00:12<00:00, 11.4MiB/s]
/Users/mathieubartozzi/.pyenv/versions/3.10.6/envs/video_auto_rating/lib/python3.10/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please op

In [12]:
# Chemin vers le fichier vidéo à tester (place une vidéo dans le dossier "data/videos")
video_folder = "../data/videos/"

In [28]:
# Fonction pour extraire les noms et prénoms à partir des fichiers vidéos

def extract_names_from_videos(video_folder):
    video_files = [f for f in os.listdir(video_folder) if f.endswith(('.mp4', '.avi', '.mov'))]

    # Séparer le nom et le prénom en supposant que le format est "NOM_prénom.xxx"
    data = []
    for video in video_files:
        base_name = os.path.splitext(video)[0]  # Enlever l'extension (.mp4, .avi, etc.)
        if '_' in base_name:
            nom, prenom = base_name.split('_', 1)  # Séparer sur le premier '_'
        else:
            nom, prenom = base_name, ''  # Si pas de '_', mettre prénom vide

        data.append({'video_file': video, 'nom': nom, 'prénom': prenom})

    # Créer un DataFrame
    df_videos = pd.DataFrame(data)
    return df_videos

df_videos=extract_names_from_videos(video_folder)

## Extraction du transcript

In [29]:
# Spécifie le modèle Whisper à utiliser
whisper_model_name = "base"  # Par exemple, "tiny", "base", "small", "medium", ou "large"
whisper_model = whisper.load_model(whisper_model_name)

/Users/mathieubartozzi/.pyenv/versions/3.10.6/envs/video_auto_rating/lib/python3.10/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  chec

In [30]:
# Fonction pour transcrire une vidéo avec Whisper
def transcribe_video(whisper_model, video_path):
    result = whisper_model.transcribe(video_path)
    return result['text']

# Ajouter une colonne 'transcript' avec les transcriptions de chaque vidéo
df_videos['transcript'] = df_videos['video_file'].apply(
    lambda video_file: transcribe_video(whisper_model, os.path.join(video_folder, video_file))
)

/Users/mathieubartozzi/.pyenv/versions/3.10.6/envs/video_auto_rating/lib/python3.10/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [31]:
df_videos.head(5)

,video_file,nom,prénom,transcript
0,BETHOUX_camille.mp4,BETHOUX,camille,"Bonjour, je vais répondre à l'offre de Synaxi..."


## Analyse non-verbale avec Mediapipe

In [32]:
import cv2
import mediapipe as mp

# Initialiser Mediapipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# Initialiser Mediapipe Drawing pour dessiner les repères sur l'image
mp_drawing = mp.solutions.drawing_utils

Matplotlib is building the font cache; this may take a moment.
I0000 00:00:1729780755.817887 3262143 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1729780756.223282 3309884 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1729780756.270137 3309888 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [34]:
# Fonction pour analyser une vidéo et extraire les informations non-verbales
def analyze_video(video_path):
    cap = cv2.VideoCapture(video_path)  # Ouvrir la vidéo
    pose_data = []  # Stocker les résultats

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # Fin de la vidéo

        # Convertir l'image en format RGB pour Mediapipe
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Obtenir les résultats de pose
        results = pose.process(rgb_frame)

        # Vérifier si des poses sont détectées
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            pose_data.append(landmarks)

            # Dessiner les landmarks de la pose sur l'image (facultatif pour visualisation)
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    cap.release()
    return pose_data  # Retourner les informations de pose


# Exemple pour ajouter les données de posture dans le DataFrame
df_videos['pose_data'] = df_videos['video_file'].apply(
    lambda video_file: analyze_video(os.path.join(video_folder, video_file))
)

W0000 00:00:1729780848.479720 3309882 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
/Users/mathieubartozzi/.pyenv/versions/3.10.6/envs/video_auto_rating/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [37]:
df_videos.head(5)

,video_file,nom,prénom,transcript,pose_data
0,BETHOUX_camille.mp4,BETHOUX,camille,"Bonjour, je vais répondre à l'offre de Synaxi...",[[x: 0.576544762\ny: 0.380587429\nz: -1.300475...


## Mesurer le débit de parole

In [39]:
import cv2

# Fonction pour calculer la durée d'une vidéo en minutes
def get_video_duration(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0
    fps = cap.get(cv2.CAP_PROP_FPS)  # Taux de frames par seconde
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Nombre total de frames
    duration_seconds = frame_count / fps  # Durée en secondes
    cap.release()
    return duration_seconds / 60  # Convertir en minutes

# Fonction pour compter les mots dans une transcription
def count_words(transcript):
    return len(transcript.split())

# Calculer le débit de parole (mots par minute)
def calculate_speech_rate(transcript, video_duration_minutes):
    word_count = count_words(transcript)
    if video_duration_minutes > 0:
        return word_count / video_duration_minutes  # Mots par minute
    else:
        return 0

# Ajouter une colonne 'speech_rate' avec le débit de parole
df_videos['speech_rate'] = df_videos.apply(
    lambda row: calculate_speech_rate(row['transcript'], get_video_duration(os.path.join(video_folder, row['video_file']))),
    axis=1
)


In [40]:
df_videos.head(5)

,video_file,nom,prénom,transcript,pose_data,speech_rate
0,BETHOUX_camille.mp4,BETHOUX,camille,"Bonjour, je vais répondre à l'offre de Synaxi...",[[x: 0.576544762\ny: 0.380587429\nz: -1.300475...,127.208481


## Mesure du ton

In [62]:
import subprocess
import librosa
import numpy as np

audio_folder = "../data/audios"


# Fonction pour extraire l'audio d'une vidéo et l'enregistrer dans le dossier audios
def extract_audio(video_path, output_audio_path):
    command = f"ffmpeg -i {video_path} -q:a 0 -map a {output_audio_path} -y"
    subprocess.call(command, shell=True)


In [67]:
def analyze_pitch_variation(audio_path):
    # Charger l'audio avec Librosa
    y, sr = librosa.load(audio_path)

    # Ajuster la taille de la fenêtre et le hop_length
    frame_length = 1024  # Taille de la fenêtre d'analyse (réduite pour s'adapter)
    hop_length = 256      # Intervalle entre deux analyses de pitch (réduit pour plus de finesse)

    # Utiliser pyin pour extraire la fréquence fondamentale (pitch)
    pitches, voiced_flag, voiced_probs = librosa.pyin(
        y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr, frame_length=frame_length, hop_length=hop_length
    )

    # Ne garder que les pitchs valides (voiced_flag = True)
    valid_pitches = pitches[~np.isnan(pitches)]

    if len(valid_pitches) == 0:
        return 0, 0  # Retourne 0 si aucun pitch n'est détecté

    # Calculer l'écart-type du pitch sur toute la vidéo
    pitch_mean = np.mean(valid_pitches)
    pitch_std = np.std(valid_pitches)

    return pitch_mean, pitch_std


In [68]:

# Ajouter une nouvelle colonne 'audio_file' en modifiant uniquement cette colonne
df_videos['audio_file'] = df_videos['video_file'].apply(
    lambda video_file: video_file.replace('.mp4', '.wav')  # Exemple : remplacer l'extension .mp4 par .wav
)

# Extraire l'audio pour chaque vidéo et l'enregistrer dans le dossier audios
for video_file, audio_file in zip(df_videos['video_file'], df_videos['audio_file']):
    extract_audio(os.path.join(video_folder, video_file), os.path.join(audio_folder, audio_file))


# Analyser les variations de pitch après l'extraction audio
df_videos['pitch_mean'], df_videos['pitch_std'] = zip(*df_videos['audio_file'].apply(
    lambda audio_file: analyze_pitch_variation(os.path.join(audio_folder, audio_file))
))

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 15.0.0 (clang-1500.3.9.4)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --enab

In [69]:
df_videos.head(5)

,video_file,nom,prénom,transcript,pose_data,speech_rate,audio_file,pitch_mean,pitch_std
0,BETHOUX_camille.mp4,BETHOUX,camille,"Bonjour, je vais répondre à l'offre de Synaxi...",[[x: 0.576544762\ny: 0.380587429\nz: -1.300475...,127.208481,BETHOUX_camille.wav,204.181449,28.33299
